# Dataset Creation Notebook

This notebook provides a template for creating and preprocessing datasets.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Configuration

In [3]:
CSV_PATH = Path("combined_unique.csv")
PROCESSED_DIR = Path("processed_datasets")
PROCESSED_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH)
df.head()

,line,isMetaphor
0,விழியில் விழுந்து இதயம் நுழைந்து,0.0
1,இரவும் பகலும் உரசிக் கொள்ளும்,0.0
2,அந்திப் பொழுதினில் வந்துவிடு,0.0
3,உன் வெள்ளிக் கொலுசொலியை,0.0
4,நீ மல்லிகைப் பூவை சூடிக் கொண்டால்,0.0


## 4. Preprocessing

In [4]:
df_processed = df[["line", "isMetaphor"]].dropna().copy()
df_processed["isMetaphor"] = df_processed["isMetaphor"].astype(int)

metaphor_df = df_processed[df_processed["isMetaphor"] == 1].copy()
non_metaphor_df = df_processed[df_processed["isMetaphor"] == 0].copy()

marker_pattern = r"போல்|போல|போன்ற"
has_marker = metaphor_df["line"].str.contains(marker_pattern, na=False)
with_marker = metaphor_df[has_marker].copy()
without_marker = metaphor_df[~has_marker].copy()

non_metaphor_df["marker_type"] = "non_metaphor"
with_marker["marker_type"] = "with_marker"
without_marker["marker_type"] = "without_marker"

df_processed = pd.concat(
    [non_metaphor_df, with_marker, without_marker],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

In [10]:
len(non_metaphor_df), len(with_marker), len(without_marker)

(6679, 4250, 1466)

## 5. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
RANDOM_STATE = 42

def split_stratified(dataset, stratify_col):
    train_df, temp_df = train_test_split(
        dataset,
        test_size=(1 - TRAIN_RATIO),
        random_state=RANDOM_STATE,
        stratify=dataset[stratify_col]
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        random_state=RANDOM_STATE,
        stratify=temp_df[stratify_col]
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

def build_binary_dataset(positive_df, negative_df, positive_name, negative_multiplier=1):
    positive_df = positive_df.copy()
    negative_df = negative_df.copy()

    positive_df["label"] = 1
    positive_df["dataset_type"] = positive_name
    negative_df["label"] = 0
    negative_df["dataset_type"] = "non_metaphor"

    dataset = pd.concat([positive_df, negative_df], ignore_index=True)
    dataset = dataset[["line", "label", "isMetaphor", "marker_type", "dataset_type"]]

    train_df, val_df, test_df = split_stratified(dataset, "label")

    train_pos = train_df[train_df["label"] == 1]
    train_neg = train_df[train_df["label"] == 0]
    target_neg_count = min(len(train_neg), len(train_pos) * negative_multiplier)
    train_neg_sampled = train_neg.sample(n=target_neg_count, random_state=RANDOM_STATE)
    train_balanced = pd.concat([train_pos, train_neg_sampled], ignore_index=True)
    train_balanced = train_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    return train_balanced, val_df, test_df

def build_three_class_dataset(non_metaphor_df, with_marker, without_marker):
    three_class = pd.concat(
        [non_metaphor_df.copy(), with_marker.copy(), without_marker.copy()],
        ignore_index=True
    )
    label_map = {
        "non_metaphor": 0,
        "with_marker": 1,
        "without_marker": 2,
    }
    three_class["label"] = three_class["marker_type"].map(label_map)
    three_class["dataset_type"] = three_class["marker_type"]
    three_class = three_class[["line", "label", "isMetaphor", "marker_type", "dataset_type"]]
    return split_stratified(three_class, "label")

with_marker_train, with_marker_val, with_marker_test = build_binary_dataset(
    with_marker,
    non_metaphor_df,
    positive_name="with_marker",
    negative_multiplier=1
)

without_marker_train, without_marker_val, without_marker_test = build_binary_dataset(
    without_marker,
    non_metaphor_df,
    positive_name="without_marker",
    negative_multiplier=2
)

three_class_train, three_class_val, three_class_test = build_three_class_dataset(
    non_metaphor_df,
    with_marker,
    without_marker
)

for name, split_df in {
    "with_marker_train": with_marker_train,
    "with_marker_val": with_marker_val,
    "with_marker_test": with_marker_test,
    "without_marker_train": without_marker_train,
    "without_marker_val": without_marker_val,
    "without_marker_test": without_marker_test,
    "three_class_train": three_class_train,
    "three_class_val": three_class_val,
    "three_class_test": three_class_test,
}.items():
    print(name)
    print(split_df["label"].value_counts().sort_index())
    print()

## 6. Save Processed Dataset

In [ ]:
with_marker_train.to_csv(PROCESSED_DIR / "with_marker_train.csv", index=False)
with_marker_val.to_csv(PROCESSED_DIR / "with_marker_val.csv", index=False)
with_marker_test.to_csv(PROCESSED_DIR / "with_marker_test.csv", index=False)

without_marker_train.to_csv(PROCESSED_DIR / "without_marker_train.csv", index=False)
without_marker_val.to_csv(PROCESSED_DIR / "without_marker_val.csv", index=False)
without_marker_test.to_csv(PROCESSED_DIR / "without_marker_test.csv", index=False)

three_class_train.to_csv(PROCESSED_DIR / "three_class_train.csv", index=False)
three_class_val.to_csv(PROCESSED_DIR / "three_class_val.csv", index=False)
three_class_test.to_csv(PROCESSED_DIR / "three_class_test.csv", index=False)

print("Dataset saved successfully.")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(path)